# Module 2.3: Semantic Memory

The previous notebook showed that **episodic memory** lets the agent
store significant events — things that *happened*: "booked Marriott
last Tuesday", "rated the NYC trip 5 stars", "flight was delayed 2 hours."

But what about facts that aren't events?

- "I prefer Marriott" — that's not something that happened, it's a
  standing preference
- "I'm a Senior Engineer" — that's a profile fact, not an event
- "Seniors have a $300/night budget" — that's a policy constraint

Episodic memory has no place for these. They don't fit `{event_type,
description, timestamp}` because they didn't *happen* — they're
persistent truths about the world.

| Chat History (2.1) | Episodic Memory (2.2) | Semantic Memory (2.3) |
|---|---|---|

| Remembers the conversation | Stores what **happened** | Stores what is **true** |stores facts as connected triplets and enables multi-hop reasoning.

| Forgets between sessions | Recalls past events | Connects facts via relationships |This notebook introduces **semantic memory** — a knowledge graph that

| Can't personalize | Personalizes from history | Reasons across connected knowledge |

## Gap 1: No Connections (Why We Need a Graph)

Even if you stretch episodic memory to store preferences as "events",
you get flat, disconnected documents:
```json
{"event_type": "trip",     "description": "Stayed at Marriott Midtown, $280/night"}
{"event_type": "feedback", "description": "Rated Marriott Midtown 5 stars"}
```

But the facts that tie these together — Sarah's level, her budget,
her preference for Marriott — aren't events at all. They're standing
truths that need a different representation.

The agent cannot answer: *"Given her level, preferences, AND budget —
what hotel should she book in NYC?"*

The answer requires **traversing connections** between facts:

```
Sarah → PREFERS → Marriott
Sarah → HAS_LEVEL → Senior
Senior → MAX_BUDGET → $300/night
Marriott Midtown → LOCATED_IN → NYC
Marriott Midtown → PRICE → $280/night
```

This is a **knowledge graph** — facts stored as triplets:
`(Subject) -[Relationship]→ (Object)`

```mermaid
graph LR
  S[Sarah] -->|PREFERS| M[Marriott]
  S -->|HAS_LEVEL| Sr[Senior]
  S -->|PREFERS_SEAT| A[Aisle]
  S -->|FLIES_WITH| U[United]
  Sr -->|MAX_BUDGET| B[$300/night]
  M -->|LOCATED_IN| NYC[New York]
```

Each node is an **entity** (person, organization, location). Each edge
is a **relationship**. The graph is traversable — the agent can walk
from Sarah → through her preferences → to budget constraints → to matching hotels. 
Multi-hop reasoning becomes a graph query.`neo4j-agent-memory` SDK (Neo4j Labs' production library for agent memory).

We use **Neo4j** — the most widely adopted graph database

## Gap 2: Infinite Schema (The Harder Problem)

Great — we have a graph. But how do we **populate** it?

When Sarah says her airline preference, she might phrase it as:
- "I like United" → `(Sarah) -[LIKES]→ (United)`
- "I prefer United for domestic" → `(Sarah) -[PREFERS]→ (United)`
- "I always fly United" → `(Sarah) -[FLIES_WITH]→ (United)`

Three different relationship types for the **same fact**. If you define
the schema upfront (only allow `PREFERS`), new phrasings break it. If
you allow free-form relationship names, recall becomes impossible —
do you search for `LIKES`? `PREFERS`? `FLIES_WITH`?

This is the **infinite schema problem**: you cannot predefine every way
a user will express a preference.

The solution requires:
1. An **extractor** smart enough to normalize at write time
2. A **recall mechanism** that works regardless of original phrasing

The `neo4j-agent-memory` SDK solves both.

## The Solution

The SDK stores knowledge in three buckets:

| Bucket | What it stores | How it's queried |
|---|---|---|
| **Entities** | Typed nodes (Person, Org, Location) | Fuzzy + embedding match |
| **Relationships** | Edges between entities | Graph traversal |
| **Preferences** | Natural language + vector embedding | **Semantic similarity** |

The key insight: **preferences are stored as natural language with
vector embeddings**. When the agent recalls, it uses vector similarity —
so "what airline?" finds "Prefers United for domestic flights" regardless
of how it was originally phrased. No keyword matching, no rigid schema.

**Write path**: `LLMEntityExtractor` automatically extracts entities,
relationships, and preferences from conversation text in one pass.

**Read path**: `search_preferences()` for semantic recall +
`get_related_entities()` for graph traversal.

Let's build it.

In [ ]:
%pip install -q -r ../../requirements.txt

## Prerequisites

- Everything from prior modules (Azure AI Foundry, `.env`)
- **Neo4j AuraDB** free tier (50K nodes, zero cost)
  → Follow [steps/02_setup_neo4j.md](steps/02_setup_neo4j.md)
- Add to your `.env`:
  ```
  NEO4J_URI=neo4j+s://xxxxx.databases.neo4j.io
  NEO4J_USER=neo4j
  NEO4J_PASSWORD=your-password
  ```

In [ ]:
import sys, os, asyncio, json
import nest_asyncio

sys.path.insert(0, "../..")
nest_asyncio.apply()

from dotenv import load_dotenv
from azure.identity import AzureCliCredential, get_bearer_token_provider
from shared.travel_agent import (
    create_client, SYSTEM_PROMPT,
    search_flights, search_hotels, get_travel_policy,
)

load_dotenv("../../.env", override=True)
client, credential = create_client("../../.env")
print("Foundry client ready")

In [ ]:
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.llm.adapters.openai import OpenAIProvider, OpenAIEmbeddingProvider
from pydantic import SecretStr

# Entra token — no API keys needed
token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)
endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model = os.environ.get("FOUNDRY_MODEL", "gpt-4o")

llm = OpenAIProvider(model=model, api_base=endpoint, api_key=token_provider())
embedder = OpenAIEmbeddingProvider(
    model="text-embedding-3-small", api_base=endpoint, api_key=token_provider()
)
print(f"LLM + Embeddings configured via Entra (endpoint: {endpoint})")

In [ ]:
settings = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USER"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
    llm=llm,
    embeddings=embedder,
)

memory = MemoryClient(settings)
await memory.__aenter__()
print(f"Connected to Neo4j: {os.environ['NEO4J_URI']}")

In [ ]:
from neo4j_agent_memory.extraction import LLMEntityExtractor

# SDK's built-in extractor — no custom prompts needed
extractor = LLMEntityExtractor(
    provider=llm,
    extract_relations=True,
    schema="poleo",  # Person, Organization, Location, Event, Object
)
print("LLM Entity Extractor ready (POLE+O schema, relations enabled)")

## The Three Buckets of Knowledge

The SDK organizes semantic memory into three distinct stores:

1. **Entities** — the nouns: people, organizations, locations, objects.
   Stored as typed graph nodes. Deduplicated via fuzzy + embedding match.

2. **Relationships** — the connections: `(Sarah)-[:PREFERS]->(Marriott)`.
   Stored as typed graph edges. Enable traversal queries.

3. **Preferences** — the opinions: natural language assertions stored
   with vector embeddings. This is what solves infinite schema — you
   store "Prefers United Airlines for domestic flights" as-is, and
   recall it via semantic similarity regardless of how you phrase the
   query later.

Let's see each bucket in action.

In [ ]:
# Extract entities from a natural language statement
text = "Sarah Chen is a Senior Engineer at Contoso who travels frequently to London."
result = await extractor.extract(text)

print(f"Extracted {len(result.entities)} entities:")
for e in result.entities:
    print(f"  {e.name} ({e.type}) — confidence: {e.confidence:.2f}")

if result.relationships:
    print(f"\nExtracted {len(result.relationships)} relationships:")
    for r in result.relationships:
        print(f"  ({r.subject}) -[{r.predicate}]→ ({r.object})")

In [ ]:
from neo4j_agent_memory.memory import DeduplicationConfig

dedup = DeduplicationConfig(
    auto_merge_threshold=0.92,
    use_fuzzy_matching=True,
    match_same_type_only=True,
)

# Store entities with deduplication
for entity in result.entities:
    stored, dedup_result = await memory.long_term.add_entity(
        name=entity.name, entity_type=entity.type, deduplication=dedup,
    )
    action = dedup_result.action if dedup_result else "created"
    print(f"  {stored.name} ({stored.type}) — {action}")

In [ ]:
# Store relationships as graph edges
if result.relationships:
    for rel in result.relationships:
        # Find entity IDs for subject and object
        subj_results = await memory.long_term.search_entities(query=rel.subject, limit=1)
        obj_results = await memory.long_term.search_entities(query=rel.object, limit=1)
        if subj_results and obj_results:
            await memory.long_term.add_relationship(
                from_entity_id=subj_results[0].id,
                to_entity_id=obj_results[0].id,
                relationship_type=rel.predicate,
            )
            print(f"  Edge: ({rel.subject}) -[{rel.predicate}]→ ({rel.object})")

In [ ]:
# Store preferences — natural language with vector embeddings
await memory.long_term.add_preference(
    category="airline", preference="Prefers United Airlines for domestic flights", confidence=0.9
)
await memory.long_term.add_preference(
    category="hotel", preference="Prefers Marriott hotels", confidence=0.9
)
await memory.long_term.add_preference(
    category="seating", preference="Always requests aisle seats", confidence=0.85
)
print("Stored 3 preferences (airline, hotel, seating)")

## The Infinite Schema Proof

Here's the key test. We store the **same airline preference** three
different ways — the exact scenario from Gap 2. Then we recall with a
completely different phrasing. Vector similarity handles the rest.

In [ ]:
# Store the same fact with three different phrasings
await memory.long_term.add_preference(
    category="airline", preference="I like United", confidence=0.8
)
await memory.long_term.add_preference(
    category="airline", preference="I prefer United for domestic travel", confidence=0.85
)
await memory.long_term.add_preference(
    category="airline", preference="I always fly United", confidence=0.8
)

# Recall with a DIFFERENT phrasing — vector similarity finds them all
results = await memory.long_term.search_preferences(query="what airline does Sarah use?")

print("Query: 'what airline does Sarah use?'")
print(f"Found {len(results)} matches:\n")
for r in results:
    print(f"  [{r.category}] {r.preference} (score: {r.score:.3f})")

**This is the solution to infinite schema.** The query "what airline
does Sarah use?" never appears in any stored preference. But vector
similarity finds all the relevant matches — "I like United",
"I prefer United", "I always fly United" — because they're
semantically similar.

No keyword normalization. No rigid relationship types. No schema
maintenance. The embeddings handle synonym resolution at read time.

In [ ]:
# Inspect the graph: count what's stored
entities = await memory.long_term.search_entities(query="", limit=50)
preferences = await memory.long_term.search_preferences(query="")

print(f"Graph state: {len(entities)} entities, {len(preferences)} preferences\n")
print("Entities:")
for e in entities:
    print(f"  [{e.type}] {e.name}")
print("\nPreferences:")
for p in preferences:
    print(f"  [{p.category}] {p.preference}")

## Recall: Semantic Search + Graph Traversal

Recall in a knowledge graph is not keyword search. It combines:

1. **Vector similarity** on preferences — handles phrasing variance
2. **Fuzzy + embedding match** on entities — finds "United Airlines"
   from "United"
3. **Graph traversal** on relationships — follows edges for multi-hop
   reasoning (Sarah → prefers → Marriott → located in → NYC)

In [ ]:
# Preference recall — semantic similarity, not keyword match
results = await memory.long_term.search_preferences(query="hotel choice")

print("Query: 'hotel choice'")
for r in results:
    print(f"  [{r.category}] {r.preference} (score: {r.score:.3f})")

In [ ]:
# Entity search + relationship traversal
entity_results = await memory.long_term.search_entities(query="Sarah", limit=1)

if entity_results:
    sarah = entity_results[0]
    print(f"Found: {sarah.name} ({sarah.type})")

    related = await memory.long_term.get_related_entities(entity_id=sarah.id)
    print(f"\nConnected entities ({len(related)}):")
    for rel in related:
        print(f"  -[{rel.relationship_type}]→ {rel.entity.name} ({rel.entity.type})")

In [ ]:
# Multi-hop Cypher: traverse 2 hops from Sarah
cypher = """
MATCH (s:Entity {name: 'Sarah Chen'})-[r1]->(mid)-[r2]->(target)
RETURN s.name AS source, type(r1) AS rel1, mid.name AS middle,
       type(r2) AS rel2, target.name AS destination
LIMIT 10
"""
records = await memory.graph.execute_read(cypher)
print("2-hop traversal from Sarah Chen:")
for rec in records:
    print(f"  {rec['source']} →[{rec['rel1']}]→ {rec['middle']} →[{rec['rel2']}]→ {rec['destination']}")

## Wiring Semantic Memory into the Agent

The agent needs two tools:
- **`learn_from_conversation`** — extract + store (write path)
- **`recall_knowledge`** — search + traverse (read path)

The system prompt tells the agent **when** to use each. Without it,
the agent won't know to learn from statements or recall before
recommending.

In [ ]:
from agent_framework import tool

@tool
async def learn_from_conversation(user_id: str, statement: str) -> str:
    """Extract and store facts from a user statement into the knowledge graph."""
    result = await extractor.extract(statement)
    stored = []
    for entity in result.entities:
        e, _ = await memory.long_term.add_entity(name=entity.name, entity_type=entity.type, deduplication=dedup)
        stored.append(f"Entity: {e.name}")
    for rel in result.relationships or []:
        subj = await memory.long_term.search_entities(query=rel.subject, limit=1)
        obj = await memory.long_term.search_entities(query=rel.object, limit=1)
        if subj and obj:
            await memory.long_term.add_relationship(
                from_entity_id=subj[0].id, to_entity_id=obj[0].id, relationship_type=rel.predicate)
            stored.append(f"Rel: {rel.predicate}")
    return f"Learned {len(stored)} facts: {'; '.join(stored)}"

In [ ]:
@tool
async def recall_knowledge(user_id: str, query: str) -> str:
    """Retrieve relevant knowledge from the graph via semantic search + traversal."""
    prefs = await memory.long_term.search_preferences(query=query)
    entities = await memory.long_term.search_entities(query=query, limit=5)
    parts = []
    if prefs:
        parts.append("Preferences: " + "; ".join(
            f"{p.category}: {p.preference}" for p in prefs[:5]
        ))
    if entities:
        for e in entities[:3]:
            related = await memory.long_term.get_related_entities(entity_id=e.id)
            if related:
                connections = ", ".join(f"-[{r.relationship_type}]→{r.entity.name}" for r in related[:5])
                parts.append(f"{e.name}: {connections}")
    return "\n".join(parts) if parts else f"No knowledge found for: {query}"

In [ ]:
from agent_framework import Agent

SEMANTIC_PROMPT = SYSTEM_PROMPT + "\n\n" + """You have a knowledge graph.
WHEN TO LEARN (call learn_from_conversation):
- User states a preference, fact, or constraint about themselves
- User mentions their role, level, team, or dietary needs
- User gives feedback about a hotel, airline, or destination
WHEN TO RECALL (call recall_knowledge):
- Before making any recommendation (hotels, flights, restaurants)
- When user asks about themselves or wants something personalized
Always use user_id 'E001' for Sarah Chen."""

semantic_agent = Agent(
    client=client,
    name="TravelAssistant",
    instructions=SEMANTIC_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy,
           learn_from_conversation, recall_knowledge],
)
print("Semantic agent ready (5 tools)")

## Building the Graph from Conversation

Sarah tells the agent about herself across several turns. The agent
extracts entities, relationships, and preferences into the graph.

> **Note**: Sarah identifies herself so the agent knows her `user_id`.
> In production, this comes from the auth context (Entra ID / SSO),
> not from user input.

In [ ]:
from agent_framework import AgentSession

async def sarah_shares_facts():
    session = AgentSession()
    turns = [
        "Hi, I'm Sarah Chen (E001). I'm a Senior Engineer at Contoso.",
        "I always prefer Marriott hotels and aisle seats when flying.",
        "I'm vegetarian and usually fly United. I have Star Alliance Gold.",
    ]
    for msg in turns:
        print(f"Sarah: {msg}")
        r = await semantic_agent.run(msg, session=session)
        print(f"Agent: {r.text}\n")

asyncio.run(sarah_shares_facts())

In [ ]:
# Inspect: what did the graph learn?
entities = await memory.long_term.search_entities(query="", limit=20)
preferences = await memory.long_term.search_preferences(query="")

print(f"Graph grew to: {len(entities)} entities, {len(preferences)} preferences\n")
for e in entities:
    related = await memory.long_term.get_related_entities(entity_id=e.id)
    edges = f" → {', '.join(r.entity.name for r in related)}" if related else ""
    print(f"  [{e.type}] {e.name}{edges}")

## Reasoning Across the Graph

New session — no chat history. Sarah asks for a recommendation.
The agent must **traverse the graph**: recall her preferences, combine
them with available options, and give a compound answer using facts
learned at different times in different conversations.

This is the core value of semantic memory: **connected reasoning**.

In [ ]:
async def sarah_asks_recommendation():
    session = AgentSession()  # fresh — no prior history
    turns = [
        "I'm Sarah Chen (E001). What hotel should I book in NYC given my preferences?",
        "And what flights would work for me?",
    ]
    for msg in turns:
        print(f"Sarah: {msg}")
        r = await semantic_agent.run(msg, session=session)
        print(f"Agent: {r.text}\n")

asyncio.run(sarah_asks_recommendation())

### Key Insight: Connected Reasoning

The agent combined facts learned at **different times** into one answer:
- "Prefers Marriott" (from turn 2)
- "Prefers aisle seats" (same turn)
- "Flies United" (from turn 3)
- "Star Alliance Gold" (same turn)

With episodic memory, the agent retrieves flat events and hopes the LLM
connects them. With semantic memory, the **graph encodes the connections
explicitly** — the agent traverses a structure, not a pile of documents.

## Comparison: Episodic vs Semantic Memory

| Dimension | Episodic Memory | Semantic Memory |
|---|---|---|
| **Storage** | Flat events in Cosmos DB | Graph nodes + edges in Neo4j |
| **Structure** | `{event_type, description, details}` | `(Entity)-[:REL]->(Entity)` + Preferences |
| **Query** | Filter by user + type | Vector similarity + graph traversal |
| **Schema** | Fixed event schema | Dynamic — infinite schema solved by embeddings |
| **Reasoning** | Retrieve events, hope LLM connects | Graph encodes connections explicitly |
| **Contradictions** | Both versions stored | MERGE = newer wins + dedup |
| **Best for** | "What happened?" | "What do we know?" |

## What Semantic Memory Still Cannot Do

The knowledge graph stores facts and preferences. But it cannot:

- **Execute procedures**: "How do I book an international flight
  step-by-step?" → Requires a workflow, not a fact
- **Apply dynamic rules**: "Check visa requirements, then insurance,
  then budget" → Requires conditional logic, not traversal
- **Adapt behavior by context**: "For VPs, skip the approval step" →
  Requires skill selection based on role

These require **procedural memory** — learned skills and workflows
that the agent can select and execute based on context.

### Architecture So Far

```mermaid
flowchart LR
  U[User] --> A[Agent]
  A --> T[Travel Tools]
  A --> CH[(Chat History<br/>Cosmos DB)]
  A --> EM[(Episodic Memory<br/>Cosmos DB)]
  A --> SM[(Semantic Memory<br/>Neo4j Graph)]
  SM -->|graph traversal| A
  EM -->|flat events| A
```

**Next**: Module 2.4 — Procedural Memory (skills and workflows).

In [ ]:
# Cleanup
await memory.__aexit__(None, None, None)
print("Neo4j connection closed")